<a href="https://colab.research.google.com/github/dorhoffman/SWINGPULSE/blob/main/notebooks/06_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import json
import joblib
import numpy as np
import pandas as pd

PROJECT_PATH = "/content/drive/MyDrive/SWINGPULSE"

FEATURES_PATH = (
    f"{PROJECT_PATH}/data/features/"
    "SWINGPULSE_FEATURES_DATASET.csv"
)

MODELS_FOLDER = f"{PROJECT_PATH}/models"

MODEL_PATH = (
    f"{MODELS_FOLDER}/"
    "random_forest_model.joblib"
)

METADATA_PATH = (
    f"{MODELS_FOLDER}/"
    "model_metadata.json"
)

print("Features file exists:", os.path.exists(FEATURES_PATH))
print("Model exists:", os.path.exists(MODEL_PATH))
print("Metadata exists:", os.path.exists(METADATA_PATH))

Mounted at /content/drive
Features file exists: True
Model exists: True
Metadata exists: True


In [2]:
model = joblib.load(MODEL_PATH)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as file:
    metadata = json.load(file)

FEATURE_COLUMNS = metadata["features"]
PREDICTION_HORIZON = metadata["prediction_horizon"]
TARGET_RETURN = metadata["target_return"]
DECISION_THRESHOLD = metadata["decision_threshold"]

print("Selected model:", metadata["selected_model"])
print("Decision threshold:", DECISION_THRESHOLD)
print("Prediction horizon:", PREDICTION_HORIZON)
print("Target return:", TARGET_RETURN)
print("Number of features:", len(FEATURE_COLUMNS))

Selected model: Random Forest
Decision threshold: 0.45
Prediction horizon: 10
Target return: 0.1
Number of features: 19


In [3]:
LATEST_FEATURES_PATH = (
    f"{PROJECT_PATH}/data/features/"
    "SWINGPULSE_LATEST_FEATURES.csv"
)

columns_to_load = [
    "Date",
    "Symbol",
    "Close",
    "Open",
    "High",
    "Low",
    "Volume"
] + FEATURE_COLUMNS

# הסרת כפילויות בשמות העמודות
columns_to_load = list(dict.fromkeys(columns_to_load))

print("Columns to load:", len(columns_to_load))

Columns to load: 26


In [4]:
latest_rows = {}

for chunk_number, chunk in enumerate(
    pd.read_csv(
        FEATURES_PATH,
        usecols=columns_to_load,
        chunksize=100_000,
        parse_dates=["Date"],
        low_memory=False
    ),
    start=1
):
    chunk = chunk.replace(
        [np.inf, -np.inf],
        np.nan
    )

    chunk = chunk.dropna(
        subset=["Date", "Symbol"] + FEATURE_COLUMNS
    )

    chunk = chunk.sort_values(
        ["Symbol", "Date"]
    )

    chunk_latest = (
        chunk
        .groupby("Symbol", as_index=False)
        .tail(1)
    )

    for _, row in chunk_latest.iterrows():
        symbol = row["Symbol"]

        if (
            symbol not in latest_rows
            or row["Date"] > latest_rows[symbol]["Date"]
        ):
            latest_rows[symbol] = row

    if chunk_number % 10 == 0:
        print(
            f"Processed {chunk_number} chunks | "
            f"Stocks found: {len(latest_rows)}"
        )

latest_features = pd.DataFrame(
    latest_rows.values()
)

latest_features = (
    latest_features
    .sort_values("Symbol")
    .reset_index(drop=True)
)

latest_features.to_csv(
    LATEST_FEATURES_PATH,
    index=False
)

print("\nLatest features file created")
print("Shape:", latest_features.shape)
print("Stocks:", latest_features["Symbol"].nunique())
print("Latest date:", latest_features["Date"].max())

display(latest_features.head())

Processed 10 chunks | Stocks found: 130
Processed 20 chunks | Stocks found: 260
Processed 30 chunks | Stocks found: 391

Latest features file created
Shape: (501, 26)
Stocks: 501
Latest date: 2023-09-21 04:00:00


,Date,Symbol,Open,High,Low,Close,Volume,Daily_Return,Volume_Change,SMA_20,...,MACD,MACD_Signal,MACD_Histogram,BB_Width,ATR_14,Volatility_20,Price_to_SMA20,Price_to_SMA50,Price_to_EMA20,EMA20_to_EMA50
0,2023-09-21 04:00:00,A,112.029999,112.195000,109.904999,110.220001,472462,-0.020005,-0.729233,116.574000,...,-2.690552,-2.273037,-0.417515,0.133685,3.175358,0.012821,0.945494,0.910977,0.951640,0.971728
1,2023-09-21 04:00:00,AAL,12.950000,13.250000,12.930000,13.205000,12990165,0.011103,-0.532055,14.010750,...,-0.644290,-0.615135,-0.029154,0.188390,0.367857,0.016109,0.942491,0.854587,0.946301,0.938396
2,2023-09-21 04:00:00,AAPL,174.550003,176.300003,174.498703,175.115005,27595896,-0.002137,-0.526926,180.044751,...,-1.890942,-1.566232,-0.324710,0.113242,3.890095,0.016607,0.972619,0.953670,0.979222,0.988672
3,2023-09-21 04:00:00,ABBV,153.679993,154.839996,152.710007,152.895004,1239212,-0.004655,-0.548524,149.531252,...,1.686884,1.324521,0.362362,0.075643,2.394999,0.009083,1.022495,1.035614,1.015446,1.018665
4,2023-09-21 04:00:00,ABNB,135.554993,135.979996,132.679993,133.309998,4101606,-0.034055,-0.209558,137.952501,...,1.374249,1.924262,-0.550013,0.219077,4.411714,0.024570,0.966347,0.957221,0.958903,1.019964


In [5]:
latest_features = pd.read_csv(
    LATEST_FEATURES_PATH,
    parse_dates=["Date"]
)

latest_features["Symbol"] = (
    latest_features["Symbol"]
    .astype(str)
    .str.upper()
    .str.strip()
)

print("Stocks available:", latest_features["Symbol"].nunique())
print("Latest date:", latest_features["Date"].max())

Stocks available: 501
Latest date: 2023-09-21 04:00:00


In [6]:
def interpret_rsi(rsi):
    if rsi >= 70:
        return "Overbought"
    elif rsi <= 30:
        return "Oversold"
    elif rsi >= 55:
        return "Positive momentum"
    elif rsi <= 45:
        return "Weak momentum"
    else:
        return "Neutral momentum"

In [7]:
def interpret_macd(macd, signal):
    if macd > signal:
        return "Bullish"
    elif macd < signal:
        return "Bearish"
    else:
        return "Neutral"

In [8]:
def interpret_trend(row):
    if (
        row["EMA_20"] > row["EMA_50"]
        and row["EMA_50"] > row["EMA_200"]
    ):
        return "Strong bullish trend"

    elif row["EMA_20"] > row["EMA_50"]:
        return "Short-term bullish trend"

    elif (
        row["EMA_20"] < row["EMA_50"]
        and row["EMA_50"] < row["EMA_200"]
    ):
        return "Strong bearish trend"

    else:
        return "Mixed trend"

In [9]:
def determine_signal(probability, threshold):
    if probability >= threshold + 0.15:
        return "Strong Watch"

    elif probability >= threshold:
        return "Watch"

    elif probability >= threshold - 0.10:
        return "Neutral"

    else:
        return "Low Potential"

In [10]:
def analyze_stock(symbol):
    symbol = str(symbol).upper().strip()

    stock_rows = latest_features[
        latest_features["Symbol"] == symbol
    ]

    if stock_rows.empty:
        return {
            "success": False,
            "message": (
                f"The symbol {symbol} was not found "
                "in the SWINGPULSE dataset."
            )
        }

    row = stock_rows.iloc[-1]

    stock_features = pd.DataFrame(
        [row[FEATURE_COLUMNS].to_dict()],
        columns=FEATURE_COLUMNS
    )

    stock_features = stock_features.replace(
        [np.inf, -np.inf],
        np.nan
    )

    if stock_features.isna().any().any():
        return {
            "success": False,
            "message": (
                f"Insufficient feature data for {symbol}."
            )
        }

    probability = model.predict_proba(
        stock_features
    )[0, 1]

    prediction = int(
        probability >= DECISION_THRESHOLD
    )

    rsi_status = interpret_rsi(
        row["RSI_14"]
    )

    macd_status = interpret_macd(
        row["MACD"],
        row["MACD_Signal"]
    )

    trend_status = interpret_trend(row)

    signal = determine_signal(
        probability,
        DECISION_THRESHOLD
    )

    return {
        "success": True,
        "symbol": symbol,
        "date": row["Date"],
        "close": float(row["Close"]),
        "probability": float(probability),
        "prediction": prediction,
        "signal": signal,
        "target_return": TARGET_RETURN,
        "prediction_horizon": PREDICTION_HORIZON,
        "threshold": DECISION_THRESHOLD,
        "rsi": float(row["RSI_14"]),
        "rsi_status": rsi_status,
        "macd": float(row["MACD"]),
        "macd_signal": float(row["MACD_Signal"]),
        "macd_status": macd_status,
        "trend": trend_status,
        "atr": float(row["ATR_14"]),
        "volatility": float(row["Volatility_20"]),
        "price_to_sma20": float(row["Price_to_SMA20"])
    }

In [11]:
def display_stock_analysis(symbol):
    result = analyze_stock(symbol)

    if not result["success"]:
        print(result["message"])
        return

    print("=" * 60)
    print("SWINGPULSE STOCK ANALYSIS")
    print("=" * 60)

    print(f"Symbol:              {result['symbol']}")
    print(f"Data date:           {result['date']}")
    print(f"Closing price:       ${result['close']:.2f}")

    print("\nMODEL PREDICTION")
    print("-" * 60)

    print(
        f"Probability of +{result['target_return'] * 100:.0f}% "
        f"within {result['prediction_horizon']} trading days: "
        f"{result['probability'] * 100:.2f}%"
    )

    print(
        f"Decision threshold:  "
        f"{result['threshold'] * 100:.0f}%"
    )

    print(f"Signal:              {result['signal']}")

    print("\nTECHNICAL INDICATORS")
    print("-" * 60)

    print(
        f"RSI (14):            "
        f"{result['rsi']:.2f} "
        f"({result['rsi_status']})"
    )

    print(
        f"MACD:                "
        f"{result['macd']:.4f}"
    )

    print(
        f"MACD Signal:         "
        f"{result['macd_signal']:.4f}"
    )

    print(
        f"MACD interpretation: "
        f"{result['macd_status']}"
    )

    print(f"Trend:               {result['trend']}")

    print(
        f"ATR (14):            "
        f"{result['atr']:.4f}"
    )

    print(
        f"Volatility (20):     "
        f"{result['volatility']:.4f}"
    )

    print(
        f"Price / SMA20:       "
        f"{result['price_to_sma20']:.4f}"
    )

    print("\nDISCLAIMER")
    print("-" * 60)
    print(
        "This analysis is based on historical data and a "
        "machine-learning model. It is not financial advice."
    )

In [12]:
display_stock_analysis("AAPL")

SWINGPULSE STOCK ANALYSIS
Symbol:              AAPL
Data date:           2023-09-21 04:00:00
Closing price:       $175.12

MODEL PREDICTION
------------------------------------------------------------
Probability of +10% within 10 trading days: 28.21%
Decision threshold:  45%
Signal:              Low Potential

TECHNICAL INDICATORS
------------------------------------------------------------
RSI (14):            41.64 (Weak momentum)
MACD:                -1.8909
MACD Signal:         -1.5662
MACD interpretation: Bearish
Trend:               Mixed trend
ATR (14):            3.8901
Volatility (20):     0.0166
Price / SMA20:       0.9726

DISCLAIMER
------------------------------------------------------------
This analysis is based on historical data and a machine-learning model. It is not financial advice.


In [13]:
display_stock_analysis("NVDA")

SWINGPULSE STOCK ANALYSIS
Symbol:              NVDA
Data date:           2023-09-21 04:00:00
Closing price:       $414.38

MODEL PREDICTION
------------------------------------------------------------
Probability of +10% within 10 trading days: 35.07%
Decision threshold:  45%
Signal:              Neutral

TECHNICAL INDICATORS
------------------------------------------------------------
RSI (14):            32.61 (Weak momentum)
MACD:                -7.1329
MACD Signal:         -0.6660
MACD interpretation: Bearish
Trend:               Strong bullish trend
ATR (14):            14.8448
Volatility (20):     0.0188
Price / SMA20:       0.9013

DISCLAIMER
------------------------------------------------------------
This analysis is based on historical data and a machine-learning model. It is not financial advice.


In [14]:
display_stock_analysis("NOTREAL")

The symbol NOTREAL was not found in the SWINGPULSE dataset.


In [15]:
user_symbol = input(
    "Enter a stock symbol for analysis: "
)

display_stock_analysis(user_symbol)

Enter a stock symbol for analysis: aapl
SWINGPULSE STOCK ANALYSIS
Symbol:              AAPL
Data date:           2023-09-21 04:00:00
Closing price:       $175.12

MODEL PREDICTION
------------------------------------------------------------
Probability of +10% within 10 trading days: 28.21%
Decision threshold:  45%
Signal:              Low Potential

TECHNICAL INDICATORS
------------------------------------------------------------
RSI (14):            41.64 (Weak momentum)
MACD:                -1.8909
MACD Signal:         -1.5662
MACD interpretation: Bearish
Trend:               Mixed trend
ATR (14):            3.8901
Volatility (20):     0.0166
Price / SMA20:       0.9726

DISCLAIMER
------------------------------------------------------------
This analysis is based on historical data and a machine-learning model. It is not financial advice.


In [16]:
all_stock_features = latest_features[
    FEATURE_COLUMNS
].replace(
    [np.inf, -np.inf],
    np.nan
)

valid_mask = (
    all_stock_features
    .notna()
    .all(axis=1)
)

ranking_data = latest_features.loc[
    valid_mask
].copy()

ranking_features = all_stock_features.loc[
    valid_mask
]

ranking_data["Model_Probability"] = (
    model.predict_proba(
        ranking_features
    )[:, 1]
)

ranking_data["Signal"] = ranking_data[
    "Model_Probability"
].apply(
    lambda probability: determine_signal(
        probability,
        DECISION_THRESHOLD
    )
)

In [17]:
top_opportunities = (
    ranking_data[
        [
            "Symbol",
            "Date",
            "Close",
            "RSI_14",
            "MACD",
            "Volatility_20",
            "Model_Probability",
            "Signal"
        ]
    ]
    .sort_values(
        "Model_Probability",
        ascending=False
    )
    .head(10)
    .copy()
)

top_opportunities["Probability_Percent"] = (
    top_opportunities["Model_Probability"]
    .mul(100)
    .round(2)
)

display(
    top_opportunities[
        [
            "Symbol",
            "Date",
            "Close",
            "Probability_Percent",
            "Signal",
            "RSI_14",
            "MACD",
            "Volatility_20"
        ]
    ]
)

,Symbol,Date,Close,Probability_Percent,Signal,RSI_14,MACD,Volatility_20
356,PARA,2023-09-21 04:00:00,13.495000,68.02,Strong Watch,42.486968,-0.401199,0.030469
477,WBD,2023-09-21 04:00:00,11.625000,65.18,Strong Watch,42.876926,-0.338043,0.034294
196,FSLR,2023-09-21 04:00:00,166.225006,58.67,Watch,35.607981,-5.631035,0.024205
262,KEY,2023-09-21 04:00:00,10.850000,54.15,Watch,44.599632,0.058543,0.023011
499,ZION,2023-09-21 04:00:00,34.440102,53.58,Watch,45.043301,0.175107,0.025745
239,ILMN,2023-09-21 04:00:00,135.464996,53.26,Watch,17.390177,-9.220524,0.019554
150,DXCM,2023-09-21 04:00:00,89.540001,52.10,Watch,27.267789,-4.989842,0.026637
421,STX,2023-09-21 04:00:00,66.080002,51.19,Watch,51.753321,-0.174344,0.033172
172,ETSY,2023-09-21 04:00:00,64.629997,51.15,Watch,32.674141,-4.022455,0.022115
498,ZBRA,2023-09-21 04:00:00,231.774994,49.95,Watch,26.360633,-8.368317,0.019585


In [19]:
display_stock_analysis("AAPL")

SWINGPULSE STOCK ANALYSIS
Symbol:              AAPL
Data date:           2023-09-21 04:00:00
Closing price:       $175.12

MODEL PREDICTION
------------------------------------------------------------
Probability of +10% within 10 trading days: 28.21%
Decision threshold:  45%
Signal:              Low Potential

TECHNICAL INDICATORS
------------------------------------------------------------
RSI (14):            41.64 (Weak momentum)
MACD:                -1.8909
MACD Signal:         -1.5662
MACD interpretation: Bearish
Trend:               Mixed trend
ATR (14):            3.8901
Volatility (20):     0.0166
Price / SMA20:       0.9726

DISCLAIMER
------------------------------------------------------------
This analysis is based on historical data and a machine-learning model. It is not financial advice.


In [20]:
ranking_features = latest_features[FEATURE_COLUMNS].copy()

ranking_features = ranking_features.replace(
    [np.inf, -np.inf],
    np.nan
)

valid_rows = ranking_features.notna().all(axis=1)

ranking_data = latest_features.loc[valid_rows].copy()

ranking_features = ranking_features.loc[valid_rows]

ranking_data["Probability"] = (
    model.predict_proba(ranking_features)[:,1]
)

ranking_data["Prediction"] = (
    ranking_data["Probability"] >= DECISION_THRESHOLD
).astype(int)

In [21]:
ranking_data["Score"] = (
    ranking_data["Probability"]*100
).round(2)

In [22]:
ranking_data = ranking_data.sort_values(
    "Probability",
    ascending=False
).reset_index(drop=True)

In [23]:
top20 = ranking_data[
    [
        "Symbol",
        "Date",
        "Close",
        "Probability",
        "Score",
        "Prediction",
        "RSI_14",
        "MACD",
        "ATR_14",
        "Volatility_20"
    ]
].head(20)

display(top20)

,Symbol,Date,Close,Probability,Score,Prediction,RSI_14,MACD,ATR_14,Volatility_20
0,PARA,2023-09-21 04:00:00,13.495000,0.680225,68.02,1,42.486968,-0.401199,0.623851,0.030469
1,WBD,2023-09-21 04:00:00,11.625000,0.651846,65.18,1,42.876926,-0.338043,0.563214,0.034294
2,FSLR,2023-09-21 04:00:00,166.225006,0.586687,58.67,1,35.607981,-5.631035,7.734285,0.024205
3,KEY,2023-09-21 04:00:00,10.850000,0.541471,54.15,1,44.599632,0.058543,0.426071,0.023011
4,ZION,2023-09-21 04:00:00,34.440102,0.535827,53.58,1,45.043301,0.175107,1.413571,0.025745
5,ILMN,2023-09-21 04:00:00,135.464996,0.532573,53.26,1,17.390177,-9.220524,5.237859,0.019554
6,DXCM,2023-09-21 04:00:00,89.540001,0.521018,52.10,1,27.267789,-4.989842,4.059284,0.026637
7,STX,2023-09-21 04:00:00,66.080002,0.511908,51.19,1,51.753321,-0.174344,2.431765,0.033172
8,ETSY,2023-09-21 04:00:00,64.629997,0.511459,51.15,1,32.674141,-4.022455,2.529357,0.022115
9,ZBRA,2023-09-21 04:00:00,231.774994,0.499456,49.95,1,26.360633,-8.368317,6.506429,0.019585


In [24]:
def signal(prob):

    if prob>=0.75:
        return "⭐⭐⭐⭐⭐"

    elif prob>=0.65:
        return "⭐⭐⭐⭐"

    elif prob>=0.55:
        return "⭐⭐⭐"

    elif prob>=0.45:
        return "⭐⭐"

    return "⭐"

In [25]:
ranking_data["Rating"] = ranking_data["Probability"].apply(signal)

In [26]:
display(

ranking_data[
[
"Rating",
"Symbol",
"Probability",
"Close",
"RSI_14",
"MACD",
"Volatility_20"
]

].head(20)

.style.format({

"Probability":"{:.2%}",
"Close":"${:.2f}",
"RSI_14":"{:.1f}",
"MACD":"{:.2f}",
"Volatility_20":"{:.2%}"

})

)

,Rating,Symbol,Probability,Close,RSI_14,MACD,Volatility_20
0,⭐⭐⭐⭐,PARA,68.02%,$13.49,42.5,-0.40,3.05%
1,⭐⭐⭐⭐,WBD,65.18%,$11.62,42.9,-0.34,3.43%
2,⭐⭐⭐,FSLR,58.67%,$166.23,35.6,-5.63,2.42%
3,⭐⭐,KEY,54.15%,$10.85,44.6,0.06,2.30%
4,⭐⭐,ZION,53.58%,$34.44,45.0,0.18,2.57%
5,⭐⭐,ILMN,53.26%,$135.46,17.4,-9.22,1.96%
6,⭐⭐,DXCM,52.10%,$89.54,27.3,-4.99,2.66%
7,⭐⭐,STX,51.19%,$66.08,51.8,-0.17,3.32%
8,⭐⭐,ETSY,51.15%,$64.63,32.7,-4.02,2.21%
9,⭐⭐,ZBRA,49.95%,$231.77,26.4,-8.37,1.96%


In [27]:
def AI(symbol):
    display_stock_analysis(symbol)

In [28]:
ranking_features = (
    latest_features[FEATURE_COLUMNS]
    .replace([np.inf, -np.inf], np.nan)
)

valid_mask = ranking_features.notna().all(axis=1)

market_ranking = latest_features.loc[
    valid_mask
].copy()

market_ranking["Probability"] = model.predict_proba(
    ranking_features.loc[valid_mask]
)[:, 1]

market_ranking["Signal"] = market_ranking[
    "Probability"
].apply(
    lambda probability: determine_signal(
        probability,
        DECISION_THRESHOLD
    )
)

market_ranking = (
    market_ranking
    .sort_values(
        "Probability",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Stocks ranked:", len(market_ranking))

Stocks ranked: 501


In [29]:
def show_top_opportunities(limit=10):
    limit = max(1, min(int(limit), 50))

    results = market_ranking.head(limit).copy()

    results["Probability_Percent"] = (
        results["Probability"]
        .mul(100)
        .round(2)
    )

    display(
        results[
            [
                "Symbol",
                "Date",
                "Close",
                "Probability_Percent",
                "Signal",
                "RSI_14",
                "MACD",
                "Volatility_20"
            ]
        ].style.format({
            "Close": "${:.2f}",
            "Probability_Percent": "{:.2f}%",
            "RSI_14": "{:.2f}",
            "MACD": "{:.4f}",
            "Volatility_20": "{:.4f}"
        })
    )

In [30]:
show_top_opportunities(10)

,Symbol,Date,Close,Probability_Percent,Signal,RSI_14,MACD,Volatility_20
0,PARA,2023-09-21 04:00:00,$13.49,68.02%,Strong Watch,42.49,-0.4012,0.0305
1,WBD,2023-09-21 04:00:00,$11.62,65.18%,Strong Watch,42.88,-0.3380,0.0343
2,FSLR,2023-09-21 04:00:00,$166.23,58.67%,Watch,35.61,-5.6310,0.0242
3,KEY,2023-09-21 04:00:00,$10.85,54.15%,Watch,44.60,0.0585,0.0230
4,ZION,2023-09-21 04:00:00,$34.44,53.58%,Watch,45.04,0.1751,0.0257
5,ILMN,2023-09-21 04:00:00,$135.46,53.26%,Watch,17.39,-9.2205,0.0196
6,DXCM,2023-09-21 04:00:00,$89.54,52.10%,Watch,27.27,-4.9898,0.0266
7,STX,2023-09-21 04:00:00,$66.08,51.19%,Watch,51.75,-0.1743,0.0332
8,ETSY,2023-09-21 04:00:00,$64.63,51.15%,Watch,32.67,-4.0225,0.0221
9,ZBRA,2023-09-21 04:00:00,$231.77,49.95%,Watch,26.36,-8.3683,0.0196


In [31]:
def show_stocks_above_probability(
    minimum_probability,
    limit=20
):
    probability = float(minimum_probability)

    # מאפשר להזין 60 במקום 0.60
    if probability > 1:
        probability = probability / 100

    if not 0 <= probability <= 1:
        print("Probability must be between 0 and 1, or 0 and 100.")
        return

    results = market_ranking[
        market_ranking["Probability"] >= probability
    ].head(limit).copy()

    if results.empty:
        print(
            f"No stocks were found with probability "
            f"above {probability * 100:.0f}%."
        )
        return

    results["Probability_Percent"] = (
        results["Probability"]
        .mul(100)
        .round(2)
    )

    print(
        f"Stocks with model probability of at least "
        f"{probability * 100:.0f}%:"
    )

    display(
        results[
            [
                "Symbol",
                "Date",
                "Close",
                "Probability_Percent",
                "Signal",
                "RSI_14",
                "MACD",
                "Volatility_20"
            ]
        ].style.format({
            "Close": "${:.2f}",
            "Probability_Percent": "{:.2f}%",
            "RSI_14": "{:.2f}",
            "MACD": "{:.4f}",
            "Volatility_20": "{:.4f}"
        })
    )

In [32]:
show_stocks_above_probability(60)

Stocks with model probability of at least 60%:


,Symbol,Date,Close,Probability_Percent,Signal,RSI_14,MACD,Volatility_20
0,PARA,2023-09-21 04:00:00,$13.49,68.02%,Strong Watch,42.49,-0.4012,0.0305
1,WBD,2023-09-21 04:00:00,$11.62,65.18%,Strong Watch,42.88,-0.3380,0.0343


In [33]:
def compare_stocks(symbol_1, symbol_2):
    result_1 = analyze_stock(symbol_1)
    result_2 = analyze_stock(symbol_2)

    errors = []

    if not result_1["success"]:
        errors.append(result_1["message"])

    if not result_2["success"]:
        errors.append(result_2["message"])

    if errors:
        for error in errors:
            print(error)
        return

    comparison = pd.DataFrame([
        {
            "Symbol": result_1["symbol"],
            "Date": result_1["date"],
            "Close": result_1["close"],
            "Probability": result_1["probability"],
            "Signal": result_1["signal"],
            "RSI": result_1["rsi"],
            "MACD": result_1["macd"],
            "MACD_Status": result_1["macd_status"],
            "Trend": result_1["trend"],
            "ATR": result_1["atr"],
            "Volatility": result_1["volatility"]
        },
        {
            "Symbol": result_2["symbol"],
            "Date": result_2["date"],
            "Close": result_2["close"],
            "Probability": result_2["probability"],
            "Signal": result_2["signal"],
            "RSI": result_2["rsi"],
            "MACD": result_2["macd"],
            "MACD_Status": result_2["macd_status"],
            "Trend": result_2["trend"],
            "ATR": result_2["atr"],
            "Volatility": result_2["volatility"]
        }
    ])

    comparison["Probability_Percent"] = (
        comparison["Probability"]
        .mul(100)
        .round(2)
    )

    display(
        comparison[
            [
                "Symbol",
                "Date",
                "Close",
                "Probability_Percent",
                "Signal",
                "RSI",
                "MACD_Status",
                "Trend",
                "ATR",
                "Volatility"
            ]
        ].style.format({
            "Close": "${:.2f}",
            "Probability_Percent": "{:.2f}%",
            "RSI": "{:.2f}",
            "ATR": "{:.4f}",
            "Volatility": "{:.4f}"
        })
    )

    better_stock = comparison.sort_values(
        "Probability",
        ascending=False
    ).iloc[0]

    print(
        f"\nAccording to the model, "
        f"{better_stock['Symbol']} has the higher probability "
        f"of gaining at least {TARGET_RETURN * 100:.0f}% "
        f"within {PREDICTION_HORIZON} trading days."
    )

In [34]:
compare_stocks("NVDA", "AMD")

,Symbol,Date,Close,Probability_Percent,Signal,RSI,MACD_Status,Trend,ATR,Volatility
0,NVDA,2023-09-21 04:00:00,$414.38,35.07%,Neutral,32.61,Bearish,Strong bullish trend,14.8448,0.0188
1,AMD,2023-09-21 04:00:00,$96.61,35.72%,Neutral,33.40,Bearish,Mixed trend,3.9193,0.0254



According to the model, AMD has the higher probability of gaining at least 10% within 10 trading days.


In [35]:
import re


def swingpulse_assistant(user_request):
    request = str(user_request).strip()
    request_lower = request.lower()

    if not request:
        print("Please enter a request.")
        return

    # Show today's top opportunities
    if (
        "top opportunities" in request_lower
        or "top stocks" in request_lower
        or "best opportunities" in request_lower
    ):
        number_match = re.search(
            r"\b(\d{1,2})\b",
            request_lower
        )

        limit = (
            int(number_match.group(1))
            if number_match
            else 10
        )

        show_top_opportunities(limit)
        return

    # Show stocks with probability above 60%
    if (
        "probability above" in request_lower
        or "probability over" in request_lower
        or "probability higher than" in request_lower
    ):
        percentage_match = re.search(
            r"(\d+(?:\.\d+)?)\s*%",
            request_lower
        )

        if not percentage_match:
            print(
                "Please include a percentage, "
                "for example: probability above 60%."
            )
            return

        minimum_probability = float(
            percentage_match.group(1)
        )

        show_stocks_above_probability(
            minimum_probability
        )
        return

    # Compare NVDA and AMD
    if "compare" in request_lower:
        symbols = re.findall(
            r"\b[A-Za-z]{1,5}(?:-[A-Za-z])?\b",
            request
        )

        ignored_words = {
            "compare",
            "and",
            "with",
            "vs",
            "versus"
        }

        symbols = [
            symbol.upper()
            for symbol in symbols
            if symbol.lower() not in ignored_words
        ]

        available_symbols = set(
            latest_features["Symbol"]
            .astype(str)
            .str.upper()
        )

        symbols = [
            symbol
            for symbol in symbols
            if symbol in available_symbols
        ]

        if len(symbols) < 2:
            print(
                "Please provide two valid stock symbols, "
                "for example: Compare NVDA and AMD."
            )
            return

        compare_stocks(
            symbols[0],
            symbols[1]
        )
        return

    # Analyze AAPL
    if (
        request_lower.startswith("analyze")
        or request_lower.startswith("analyse")
    ):
        parts = request.split()

        if len(parts) < 2:
            print(
                "Please provide a symbol, "
                "for example: Analyze AAPL."
            )
            return

        symbol = parts[-1].upper()

        display_stock_analysis(symbol)
        return

    print(
        "I did not understand the request.\n\n"
        "Supported examples:\n"
        "- Analyze AAPL\n"
        "- Compare NVDA and AMD\n"
        "- Show today's top opportunities\n"
        "- Show top 5 opportunities\n"
        "- Show stocks with probability above 60%"
    )

In [36]:
user_request = input(
    "Ask SWINGPULSE: "
)

swingpulse_assistant(user_request)

Ask SWINGPULSE: Compare NVDA and AMD


,Symbol,Date,Close,Probability_Percent,Signal,RSI,MACD_Status,Trend,ATR,Volatility
0,NVDA,2023-09-21 04:00:00,$414.38,35.07%,Neutral,32.61,Bearish,Strong bullish trend,14.8448,0.0188
1,AMD,2023-09-21 04:00:00,$96.61,35.72%,Neutral,33.40,Bearish,Mixed trend,3.9193,0.0254



According to the model, AMD has the higher probability of gaining at least 10% within 10 trading days.
